<div align="center">

<a href="https://colab.research.google.com/github/utkukose/machine-ethics-ai-safety-NB-lecture/blob/main/weeks/week-04/NB04_dilemmas_social_choice.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 04: Moral Dilemmas, Pluralism and Social Choice

**Machine Ethics and Artificial Intelligence Safety (11118BLG001)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/machine-ethics-ai-safety-NB-lecture/weeks/week-04/lab.html>.

## Overview

If machines should reflect human values, whose values count and how are they combined? This week examines dilemma studies such as the Moral Machine experiment and the mathematics of aggregating preferences, including results that show why no aggregation rule is neutral [1, 2].

## Learning outcomes

By the end of the week, students are expected to interpret the findings and limitations of large dilemma studies, to compute plurality, Borda and Condorcet outcomes for a preference profile, to explain Arrow's impossibility theorem and majority cycles, and to argue when aggregation of preferences is an appropriate basis for machine behaviour.

## Dilemmas as research instruments

Moral dilemmas are simplified scenarios in which every option has a moral cost. The trolley cases of Foot and Thomson became the template [3, 4]. Vasant and Kose used dilemma scenarios to compare the possible actions of humans and AI-based systems, showing how the same scenario raises different questions when the agent is a machine [5]. Bonnefon, Shariff and Rahwan found a social dilemma in attitudes towards autonomous vehicles: Respondents approved of vehicles that would sacrifice their passengers to save more pedestrians, yet preferred to ride in vehicles that would protect their passengers [6].

Dilemmas are useful because they isolate one variable at a time. They are also artificial. Real systems rarely face a clean choice between two certain outcomes, and a system designed around dramatic dilemmas may neglect the many small trade-offs that matter more in daily operation.

## The Moral Machine experiment

Awad and colleagues deployed an online platform that presented unavoidable accident scenarios and collected about forty million decisions in ten languages from people in 233 countries and territories [1]. Strong global preferences emerged for sparing humans over animals, more lives over fewer and young lives over old. Preferences also varied across cultural clusters, which the authors linked to cultural and economic variables.

The study is a landmark and a cautionary tale. Its participants were self-selected, its scenarios were stylised and it measured stated rather than revealed preferences. More fundamentally, a majority preference is not automatically a justified norm. A preference for sparing people of higher status, for example, raises questions of equal treatment that a vote cannot settle.

## Aggregating preferences

Noothigattu and colleagues proposed a voting-based system that learns a model of each participant's preferences from dilemma responses and aggregates the models with a voting rule when a new decision arises [7]. The approach inherits the properties of social choice theory, which studies how individual rankings can be combined.

Social choice theory contains sobering results. Condorcet observed that majority preferences can be cyclic: A majority may prefer option A to B, another majority B to C and a third C to A [8]. Arrow proved that no rule for aggregating rankings of three or more options satisfies a small set of reasonable conditions at once, namely unrestricted domain, the Pareto principle, independence of irrelevant alternatives and non-dictatorship [2]. Different rules therefore produce different winners from the same preferences. Plurality counts only first choices, the Borda count rewards broad acceptability and a Condorcet winner, when one exists, beats every rival in pairwise majority votes. Figure 4.1 shows how often no Condorcet winner exists when rankings are diverse.

![Figure 4.1](https://raw.githubusercontent.com/utkukose/machine-ethics-ai-safety-NB-lecture/main/weeks/week-04/figures/w04_fig1.png)

*Figure 4.1. Simulated share of preference profiles without a Condorcet winner when every voter ranks the alternatives independently at random. The chance of a majority cycle grows with the number of alternatives [2, 8].*

## Pluralism beyond aggregation

Aggregation treats morality as a matter of preference. Gabriel argued instead for principles that people could endorse despite their disagreements, selected through a fair process that protects minorities [9]. Moral uncertainty offers a further perspective, in which a system weighs theories rather than voters [10]. In practice, designers can combine the approaches: Hard constraints protect rights that no majority may override, and aggregation is used only among options that respect those constraints.

> **Pause and reflect.** Should the behaviour of a medical triage system deployed in Türkiye follow the preferences of Turkish respondents, of global respondents or of neither? What would justify each answer?

# Hands-on lab

The notebook implements plurality, Borda and Condorcet rules, estimates how often majority cycles occur, and aggregates synthetic dilemma preferences in the spirit of the voting-based approach of Noothigattu and colleagues [7].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. Voting rules

A profile is a list of rankings. Each ranking lists alternatives from best to worst, for example `(2, 0, 1)`.

In [ ]:
def plurality(profile, m):
    counts = np.zeros(m, dtype=int)
    for r in profile:
        counts[r[0]] += 1
    return int(np.argmax(counts)), counts

def pairwise(profile, m):
    """W[a, b] counts voters who rank a above b."""
    W = np.zeros((m, m), dtype=int)
    for r in profile:
        pos = {a: k for k, a in enumerate(r)}
        for a in range(m):
            for b in range(m):
                if a != b and pos[a] < pos[b]:
                    W[a, b] += 1
    return W

def condorcet_winner(profile, m):
    W, n = pairwise(profile, m), len(profile)
    for a in range(m):
        if all(W[a, b] > n / 2 for b in range(m) if b != a):
            return a
    return None

cycle = [(0, 1, 2), (1, 2, 0), (2, 0, 1)]
print("Condorcet winner of the classic cycle:", condorcet_winner(cycle, 3))
print(pairwise(cycle, 3))

### Your turn, exercise 1

Implement the Borda count: With m alternatives, a first place earns m - 1 points, a second place m - 2 points and so on. Return the winner and the scores.

In [ ]:
def borda(profile, m):
    scores = np.zeros(m, dtype=int)
    # Your code here
    return int(np.argmax(scores)), scores

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _borda_reference(profile, m):
    scores = np.zeros(m, dtype=int)
    for r in profile:
        for k, a in enumerate(r):
            scores[a] += m - 1 - k
    return int(np.argmax(scores)), scores

In [ ]:
test_profile = [(0, 1, 2)] * 4 + [(1, 2, 0)] * 3 + [(2, 1, 0)] * 2
check("Exercise 1", borda(test_profile, 3)[1], _borda_reference(test_profile, 3)[1])
print("plurality:", plurality(test_profile, 3)[0], " Borda:", _borda_reference(test_profile, 3)[0], " Condorcet:", condorcet_winner(test_profile, 3))

In the test profile, plurality elects alternative 0, while the Borda count and the Condorcet method elect alternative 1. The choice of rule decides the outcome.

## 2. How often do majority cycles occur?

In [ ]:
rng = np.random.default_rng(SEED)
voters = [3, 5, 9, 15, 25, 41]
for m in (3, 4, 5):
    share = []
    for n in voters:
        none = sum(condorcet_winner([tuple(rng.permutation(m)) for _ in range(n)], m) is None for _ in range(300))
        share.append(none / 300)
    plt.plot(voters, share, marker="o", label=f"{m} alternatives")
plt.xlabel("voters"); plt.ylabel("share without a Condorcet winner"); plt.legend(); plt.title("Majority cycles under random preferences")
plt.show()

For three alternatives, the simulated share approaches the classical asymptotic value of about 8.8 percent as the number of voters grows. Real preferences are more structured than random ones, so cycles are less frequent in practice, but they cannot be ruled out.

## 3. Aggregating dilemma preferences

Each synthetic respondent weighs three considerations: saving more lives, protecting pedestrians and not swerving. Four policies are ranked by each respondent, and the rankings are aggregated.

In [ ]:
POLICIES = ["minimise deaths", "protect passengers", "protect pedestrians", "never swerve"]
FEAT = np.array([[1, 0, -0.5], [0, -1, 0], [0, 1, -0.3], [-0.5, 0, 1]])

def synthetic_class(n, mean, rng):
    w = mean + rng.normal(0, 0.9, size=(n, 3))
    u = w @ FEAT.T + rng.normal(0, 0.3, size=(n, 4))
    return [tuple(np.argsort(-row)) for row in u]

disagree = 0
for trial in range(200):
    prof = synthetic_class(40, np.array([0.8, 0.2, 0.1]), rng)
    p, b, c = plurality(prof, 4)[0], _borda_reference(prof, 4)[0], condorcet_winner(prof, 4)
    disagree += len({p, b, c}) > 1
print(f"classes in which at least two rules disagree: {disagree} of 200")

### Your turn, exercise 2

Change the mean weight vector so that the class cares strongly about not swerving (for example `[0.2, 0.0, 1.0]`). Which policy wins under each rule, and does the disagreement rate change? Write three sentences on what this implies for systems that learn norms from surveys.

In [ ]:
# Your code here

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/machine-ethics-ai-safety-NB-lecture/weeks/week-04/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Did your ranking in Part B match the class winner under any rule? What does that suggest about building a system from aggregated preferences?
2. Name one preference revealed in dilemma studies that should not be implemented even if a majority holds it, and explain why.
3. How would you collect moral preferences for a system in your field so that the sample is appropriate and minorities are protected?

## Weekly task and submission

Design a small preference study for a decision made by an AI system in your field. Write four dilemma items, specify the target population and the sampling procedure, name the aggregation rule you would use and the constraints that no aggregation may override. Keep the design to about one page and attach the completed notebook.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**Aggregating moral preferences.** Analyse the Moral Machine experiment and the criticism it received [1]. Discuss whether aggregating the preferences of many people is a sound basis for machine decisions, with reference to the results of social choice theory [2].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## References

[1] Awad, E., Dsouza, S., Kim, R., Schulz, J., Henrich, J., Shariff, A., Bonnefon, J.-F., & Rahwan, I. (2018). The Moral Machine experiment. *Nature*, 563(7729), 59-64. <https://doi.org/10.1038/s41586-018-0637-6>

[2] Arrow, K. J. (1951). *Social Choice and Individual Values*. John Wiley & Sons.

[3] Foot, P. (1967). The problem of abortion and the doctrine of the double effect. *Oxford Review*, 5, 5-15.

[4] Thomson, J. J. (1985). The trolley problem. *The Yale Law Journal*, 94(6), 1395-1415. <https://doi.org/10.2307/796133>

[5] Vasant, P., & Kose, U. (2017). Moral dilemma scenarios considering possible actions by humans and artificial intelligence based systems. *Journal of Multidisciplinary Developments*, 2(2), 39-49.

[6] Bonnefon, J.-F., Shariff, A., & Rahwan, I. (2016). The social dilemma of autonomous vehicles. *Science*, 352(6293), 1573-1576. <https://doi.org/10.1126/science.aaf2654>

[7] Noothigattu, R., Gaikwad, S., Awad, E., Dsouza, S., Rahwan, I., Ravikumar, P., & Procaccia, A. D. (2018). A voting-based system for ethical decision making. In *Proceedings of the AAAI Conference on Artificial Intelligence, 32(1)*.

[8] de Condorcet, M. (1785). *Essai sur l'application de l'analyse à la probabilité des décisions rendues à la pluralité des voix*. Imprimerie Royale.

[9] Gabriel, I. (2020). Artificial intelligence, values, and alignment. *Minds and Machines*, 30(3), 411-437. <https://doi.org/10.1007/s11023-020-09539-2>

[10] MacAskill, W., Bykvist, K., & Ord, T. (2020). *Moral Uncertainty*. Oxford University Press.